# 13 — Convolutional Neural Networks: dari Operator Konvolusi hingga Eksperimen Ilmiah
### Arsitektur, optimisasi, interpretabilitas, *ablation study*, dan ketahanan model pencitraan

**Seri:** Pengolahan Citra Digital dan *Scientific Imaging* · **Bahasa:** Indonesia · **Berkas repository:** `notebooks/13_cnn_opsional.ipynb`  
**Prasyarat:** Notebook 03 (konvolusi), 06 (respons frekuensi), 08 (fitur), 09 (evaluasi ML), dan 10–12 (studi kasus).  
**Lingkungan:** Python, NumPy, scikit-learn, SciPy, Matplotlib, PyTorch; **CPU cukup**.  
**Kontrak keilmuan:** Semua angka dalam notebook adalah hasil komputasi pada dataset Digits kecil; **bukan** hasil uji kamera, pasien, atau industri.

> CNN bukan pengganti penalaran tentang data dan pembentukan citra. Ia merupakan keluarga model parametrik yang harus diuji terhadap tugas, distribusi data, dan batas komputasi tertentu.

## Peta capaian, prasyarat, dan diagnosis awal

Setelah menyelesaikan notebook, pembaca mampu:

1. Menurunkan operasi *cross-correlation* dalam `Conv2d`, dimensi tensor, jumlah parameter, dan *receptive field* efektif teoretis.
2. Menghubungkan translasi, *equivariance*, pooling, kapasitas, dan pilihan arsitektur dengan perubahan citra.
3. Menurunkan *cross-entropy*, gradien melalui *chain rule*, dan menerapkan pembaruan parameter tanpa kesalahan bentuk tensor.
4. Menjelaskan *train–validation–test*, regularisasi, augmentasi, pemilihan *checkpoint*, dan risiko *data leakage*.
5. Merancang pembandingan adil antara model klasik, MLP, CNN kecil, CNN lebih lebar, serta CNN dengan augmentasi.
6. Menilai *macro-F1*, kalibrasi, *robustness*, interpretabilitas, biaya komputasi, dan ketidakpastian statistik.

**Diagnosis awal (jawab sebelum menjalankan kode):** Mengapa jumlah parameter sebuah kernel $3\times3$ tidak bertambah saat dimensi gambar membesar? Apakah akurasi tertinggi pada data validasi menjamin generalisasi? Apa beda *feature map* dan bukti bahwa model benar-benar memakai ciri yang kita anggap penting?

**Alur:** masalah ilmiah → teori → pemeriksaan operator → rancangan eksperimen → model → validasi → test terpisah → audit hasil → refleksi dan HOTS.

## 1. Kontrak data, lingkungan, dan reproduksibilitas

Dataset yang digunakan adalah **Digits** bawaan `scikit-learn`: citra 8×8 piksel, 10 kelas angka, intensitas asli 0–16. Citra dinormalisasi dengan pembagi 16 dan dikirim ke jaringan sebagai tensor **NCHW** = `(jumlah citra, kanal, tinggi, lebar)`. Resolusi sangat kecil: hasil tidak boleh diekstrapolasi begitu saja ke dokumen kamera beresolusi tinggi.

Seluruh eksperimen memakai satu *held-out test* yang **tidak dipakai** untuk pemilihan model. Model awal, pemilihan *checkpoint*, jumlah epoch, dan keputusan augmentasi diperiksa pada validation set; data uji hanya dievaluasi setelah prosedur pemilihan selesai.

Pengulangan satu seed membuat eksperimen dapat dilacak tetapi tidak menghapus ketidakpastian dari inisialisasi dan pembagian data. Untuk praktikum CPU, kita gunakan satu pemisahan utama dan pengulangan pelatihan terkontrol; inferensi statistik dijelaskan dengan batasan yang eksplisit.

In [ ]:
import os, sys, time, copy, json, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, shift as scipy_shift
import scipy, sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (f1_score, accuracy_score, confusion_matrix, classification_report,
                             log_loss, brier_score_loss)
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 2026
np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(3, torch.get_num_threads()))
DEVICE = torch.device('cpu')  # baseline konsisten; runtime GPU bersifat opsional
plt.rcParams.update({'figure.dpi':100,'font.size':10,'axes.grid':False})
print('Python:', sys.version.split()[0], 'PyTorch:',torch.__version__)
print('sklearn:',sklearn.__version__,'SciPy:',scipy.__version__,'Device:',DEVICE)
print('Thread CPU:',torch.get_num_threads())

## 2. Pembentukan citra dan tugas pengenalan pola

Secara fisik, sensor tidak mengukur label kelas $y$. Kamera terlebih dahulu mengintegrasikan radiasi pada piksel, mengalami fungsi sebar titik (*point spread function*, PSF), dan menerima noise. Model generik sebelum digitalisasi dapat ditulis

$$
g_{i,j}=Q\!\left[ a\,(h*f)_{i,j}+b+\eta_{i,j}\right],
$$

dengan $f$ intensitas objek, $h$ PSF, $a$ penguatan, $b$ offset, $\eta$ gangguan, dan $Q$ kuantisasi. CNN selanjutnya menerima **representasi** $g$, bukan objek atau proses fisik secara langsung.

Pada Digits, kita tidak mengetahui model kamera asli secara lengkap. Karena itu, *blur*, pergeseran, dan noise pada bagian akhir merupakan **perturbasi buatan**, bukan pemodelan sensor yang telah divalidasi. Variasi pencitraan dapat mengubah input ke jaringan meskipun identitas digit tidak berubah secara semantik.

In [ ]:
digits=load_digits()
X=(digits.images.astype(np.float32)/16.0)
y=digits.target.astype(np.int64)
indices=np.arange(len(y))
idx_train,idx_temp=train_test_split(indices, test_size=0.40, random_state=SEED, stratify=y)
idx_val,idx_test=train_test_split(idx_temp, test_size=0.50, random_state=SEED, stratify=y[idx_temp])
assert not set(idx_train)&set(idx_val) and not set(idx_train)&set(idx_test) and not set(idx_val)&set(idx_test)
print('Dataset',len(X),'| train',len(idx_train),'| val',len(idx_val),'| test',len(idx_test))
fig,axes=plt.subplots(2,10,figsize=(12,3))
for k,ax in enumerate(axes.flat):
    ax.imshow(X[k],cmap='gray',vmin=0,vmax=1,interpolation='nearest'); ax.set_title(str(y[k]));ax.axis('off')
fig.suptitle('Contoh Digits (8×8 piksel); warna tampilan bukan informasi tambahan',fontsize=11)
plt.tight_layout();plt.show()

### 2.1 Unit independen dan keterbatasan dataset

Citra Digits bukan kumpulan foto berulang dengan identitas individu/sesi yang tersedia secara eksplisit dalam metadata yang digunakan di sini. Karena ID penulis atau sesi tidak dipakai, pemisahan yang kita lakukan adalah **stratified sample split**, bukan jaminan pemisahan per subjek atau perangkat. Jangan menyimpulkan kemampuan generalisasi antarpengguna.

**Kontrol kebocoran:** seluruh transformasi yang mempelajari statistik data, misalnya `StandardScaler`, harus dipasang pada training saja. Augmentasi hanya diterapkan pada batch pelatihan. Validation untuk pemilihan model, test untuk hasil akhir. Variasi stress dibuat dari test, sehingga **tidak dianggap sampel independen baru**.

In [ ]:
Xtr=np.ascontiguousarray(X[idx_train,None],dtype=np.float32)
Xva=np.ascontiguousarray(X[idx_val,None],dtype=np.float32)
Xte=np.ascontiguousarray(X[idx_test,None],dtype=np.float32)
ytr=y[idx_train]; yva=y[idx_val]; yte=y[idx_test]
print('Bentuk input training NCHW:',Xtr.shape)
print('Distribusi kelas test:',np.bincount(yte, minlength=10))
fig,ax=plt.subplots(figsize=(8,2.5));ax.bar(np.arange(10)-0.25,np.bincount(ytr,minlength=10),width=0.25,label='Train')
ax.bar(np.arange(10),np.bincount(yva,minlength=10),width=0.25,label='Validasi')
ax.bar(np.arange(10)+0.25,np.bincount(yte,minlength=10),width=0.25,label='Test')
ax.set(xlabel='Kelas angka',ylabel='Jumlah citra',title='Distribusi kelas pada pembagian terpisah');ax.legend();plt.show()

## 3. Operasi lokal dan parameter berbagi (*weight sharing*)

Pada model PyTorch, operasi yang disebut `Conv2d` secara teknis merupakan **korelasi silang**. Untuk masukan $x$ dengan $C_{in}$ kanal dan kernel $w$ ukuran $k_h\times k_w$, keluaran kanal ke-$o$ adalah

$$
z_{o,i,j}=b_o+\sum_{c=1}^{C_{in}}\sum_{u=0}^{k_h-1}\sum_{v=0}^{k_w-1}
w_{o,c,u,v}\,x_{c,\,is_h+u d_h-p_h,\,js_w+v d_w-p_w}.
$$

Indeks di luar citra mengikuti aturan padding. Penulisan ini menggunakan konvensi PyTorch tanpa membalik kernel; konvolusi matematis membalik kernel pada kedua arah spasial. Untuk $C_{out}$ kernel, jumlah parameternya adalah

$$
N_\theta=C_{out}\big(C_{in}k_h k_w+1\big)
$$

apabila bias diaktifkan. *Weight sharing* membuat jumlah parameter lapisan tidak bergantung langsung pada luas citra, tetapi biaya operasi tetap bertambah ketika resolusi naik.

Ukuran satu dimensi keluaran dihitung melalui

$$
L_{out}=\left\lfloor\frac{L_{in}+2p-d(k-1)-1}{s}+1\right\rfloor.
$$

In [ ]:
# Verifikasi operasi Conv2d dengan penjumlahan manual di satu lokasi.
contoh=torch.arange(1,17,dtype=torch.float32).reshape(1,1,4,4)
kernel=torch.tensor([[[[1.,0.],[-1.,2.]]]])
hasil=F.conv2d(contoh,kernel)
manual=sum(float(kernel[0,0,u,v]*contoh[0,0,u,v]) for u in range(2) for v in range(2))
assert np.isclose(float(hasil[0,0,0,0]),manual)
fig,axs=plt.subplots(1,3,figsize=(9,3))
for ax,data,t in zip(axs,[contoh[0,0],kernel[0,0],hasil[0,0]],['Input 4×4','Kernel 2×2','Output valid 3×3']):
    ax.imshow(data,cmap='coolwarm');ax.set_title(t);ax.axis('off')
    for (i,j),v in np.ndenumerate(data.numpy()):ax.text(j,i,f'{v:.0f}',ha='center',va='center',fontsize=9)
plt.tight_layout();plt.show()
print('Korelasi manual (0,0):',manual,'| Conv2d:',float(hasil[0,0,0,0]))

### 3.1 Equivariance bukan invariance

Dengan padding dan pengabaian efek tepi, korelasi konvolusional dengan stride 1 bersifat *translation equivariant*: penggeseran input menggeser respons fitur. Hal tersebut **tidak berarti** keluaran kelas selalu tidak berubah. *Pooling*, stride, padding, pemotongan, dan lapisan akhir dapat menghasilkan sensitivitas tertentu terhadap translasi. Kita perlu mengujinya, bukan mengasumsikannya.

Operator pooling maksimum bersifat nonlinear. Ia dapat mempertahankan keberadaan respons kuat, tetapi membuang posisi dan amplitudo lain yang mungkin penting bagi metrologi. Untuk sistem yang mengukur diameter atau lokasi tepi, invariansi yang berlebihan justru dapat merusak informasi yang dibutuhkan.

In [ ]:
img=np.zeros((11,11),dtype=np.float32);img[5,5]=1
w=np.ones((1,1,3,3),dtype=np.float32)/9
inp=torch.tensor(img[None,None]);inp_s=torch.tensor(np.roll(img,1,axis=1).copy()[None,None])
y1=F.conv2d(inp,torch.tensor(w),padding=1)[0,0].numpy()
y2=F.conv2d(inp_s,torch.tensor(w),padding=1)[0,0].numpy()
err=np.abs(y2-np.roll(y1,1,axis=1)).max()
fig,axs=plt.subplots(1,3,figsize=(9,2.8))
for ax,data,t in zip(axs,[img,y1,y2],['Impuls','Respons filter','Respons setelah geser']):ax.imshow(data,vmin=0,vmax=.14,cmap='magma');ax.set_title(t);ax.axis('off')
plt.tight_layout();plt.show()
print('Galat equivariance (contoh jauh dari batas):',float(err))
assert err<1e-6

## 4. Receptive field, kapasitas, dan arti satu *feature map*

Untuk lapisan berurutan, *receptive field* teoritis $r_l$ dan langkah efektif $j_l$ memenuhi

$$
r_l=r_{l-1}+(k_l-1)d_l j_{l-1},\qquad j_l=s_l j_{l-1},
$$

dengan $r_0=j_0=1$. Operator kernel 3, pooling 2, kernel 3 memiliki $r=8$ pada keluaran lapisan kedua jika stride yang digunakan masing-masing $1,2,1$. Ini bukan bukti bahwa semua piksel pada bidang tersebut berkontribusi dengan bobot sama: *effective receptive field* bergantung pada bobot dan data.

CNN membentuk hierarki fitur: respons garis, kombinasi sudut, lalu pola yang diskriminatif **jika** pelatihan menemukan parameter tersebut. Tidak ada jaminan neuron akan merepresentasikan objek sesuai istilah manusia.

In [ ]:
rf=1; jump=1
history=[]
for nama,k,s,d in [('Conv3×3',3,1,1),('Pool2×2',2,2,1),('Conv3×3',3,1,1)]:
    rf=rf+(k-1)*d*jump; jump*=s
    history.append((nama,rf,jump))
print(pd.DataFrame(history,columns=['Lapisan','Receptive field teoretis (px)','Langkah efektif (px)']).to_string(index=False))
assert history[-1][1]==8

## 5. Dari logit ke probabilitas dan fungsi kerugian

Keluaran mentah jaringan adalah *logit* $z_k\in\mathbb{R}$, bukan probabilitas. Softmax memberi

$$
p_k=\frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}},\qquad \sum_kp_k=1.
$$

Untuk label satu kelas $t$ dan target *one-hot* $y_k$, *cross-entropy* adalah

$$
\mathcal{L}_{CE}=-\sum_{k=1}^K y_k\log p_k=-\log p_t.
$$

Turunannya terhadap logit adalah

$$
\frac{\partial\mathcal{L}_{CE}}{\partial z_k}=p_k-y_k.
$$

PyTorch `nn.CrossEntropyLoss` menerima **logit**, lalu melakukan *log-softmax* secara stabil. **Jangan** melewatkan softmax terlebih dahulu ke fungsi tersebut. Gradien loss mengalir melalui semua lapisan dengan aturan rantai; optimizer mengubah parameter berdasarkan gradien dan aturan pembaruan yang dipilih.

In [ ]:
logit=torch.tensor([[1.2,-0.4,0.7]],dtype=torch.float64,requires_grad=True)
target=torch.tensor([2])
loss=F.cross_entropy(logit,target);loss.backward()
p=F.softmax(logit.detach(),dim=1).numpy().ravel()
analitik=p-np.eye(3)[2]
print('Softmax',np.round(p,5),'| loss',round(float(loss.detach()),5))
print('Gradien autograd:',np.round(logit.grad.numpy().ravel(),6))
print('Gradien analitik:',np.round(analitik,6))
assert np.allclose(logit.grad.numpy().ravel(),analitik,atol=1e-10)

## 6. *Training dynamics*: optimisasi, generalisasi, dan regularisasi

Jika parameter model $\theta$ menghasilkan prediksi $f_\theta(x)$, objektif yang digunakan dapat ditulis

$$
\min_\theta\;\frac1N\sum_{i=1}^N\mathcal{L}\!\left(f_\theta(x_i),y_i\right)+\lambda\Omega(\theta).
$$

Objektif ini membedakan ketepatan terhadap data latih dari pembatas kompleksitas. Contoh pembatas meliputi *weight decay*, dropout, pembatasan jumlah parameter, *early stopping*, dan augmentasi yang selaras dengan fenomena. AdamW menerapkan *decoupled weight decay*; ini tidak identik dengan sekadar menambahkan $\lambda\|\theta\|_2^2$ pada semua pengaturan optimizer.

Pemilihan *checkpoint* melalui validation loss adalah salah satu bentuk pemilihan model. Data validation bukan tempat mengulang eksperimen tanpa batas: terlalu banyak percobaan juga dapat menyebabkan *selection bias*.

**Catatan komputasi:** untuk menjaga praktikum dapat dijalankan pada CPU, jumlah lapisan, lebar kanal, epoch maksimum, dan ukuran batch dibatasi. Itu membuat perbandingan ini bersifat *controlled teaching experiment*, bukan pencarian arsitektur atau model terbaik secara umum.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, width=12):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(1,width,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(width,2*width,3,padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((2,2)))
        self.classifier=nn.Linear(2*width*2*2,10)
    def forward(self,x):
        return self.classifier(self.features(x).flatten(1))

class TinyMLP(nn.Module):
    def __init__(self,hidden=64):
        super().__init__()
        self.net=nn.Sequential(nn.Flatten(),nn.Linear(64,hidden),nn.ReLU(),nn.Linear(hidden,10))
    def forward(self,x):return self.net(x)

def nparams(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

for model in [TinyMLP(),TinyCNN(6),TinyCNN(12)]:
    out=model(torch.randn(5,1,8,8))
    print(type(model).__name__,'parameter:',nparams(model),'output:',tuple(out.shape))
    assert out.shape==(5,10)

### 6.1 Merancang ablation yang menjawab pertanyaan kausal terbatas

Empat konfigurasi utama dibandingkan pada **split dan protokol yang sama**:

- **SVM piksel**: baseline non-neural dengan standardisasi hanya dari train, pemilihan $C$ melalui validasi.
- **MLP**: jaringan penuh tanpa *inductive bias* konvolusional.
- **CNN kecil**: konvolusi dengan kanal lebih sedikit.
- **CNN standar**: kanal lebih banyak, lalu varian **CNN + augmentasi** dengan arsitektur identik.

Membandingkan CNN standar dengan CNN+augmentasi (parameter identik) lebih dekat pada pertanyaan dampak augmentasi dibanding membandingkan dua model yang berbeda secara serentak. Namun satu split/seed belum membuktikan efek kausal universal. Jumlah epoch maksimum dan prosedur pemilihan checkpoint diseragamkan, serta waktu komputasi dicatat sebagai informasi kontekstual, bukan benchmark lintas hardware.

In [ ]:
# Baseline nonneural: hyperparameter ditentukan hanya dari validation set.
svm_candidates=[]
for c in [0.03,0.3,3.0]:
    p=Pipeline([('scale',StandardScaler()),('clf',SVC(kernel='linear',C=c,probability=True,random_state=SEED))])
    t0=time.perf_counter()
    p.fit(Xtr.reshape(len(Xtr),-1),ytr)
    pred=p.predict(Xva.reshape(len(Xva),-1))
    score=f1_score(yva,pred,average='macro')
    svm_candidates.append((score,c,p,time.perf_counter()-t0))
    print(f'SVM C={c:g} | F1 validation={score:.4f}')
svm_candidates.sort(key=lambda t:(-t[0],t[1]))
best_svm_val, best_c, best_svm, svm_time = svm_candidates[0]
print('Terpilih dari validation: C=',best_c,'macro-F1=',round(best_svm_val,4))

### 6.2 Augmentasi yang mengubah input, bukan label

Pada citra 8×8, translasi 1 piksel adalah gangguan yang relatif besar. Augmentasi berupa translasi subpiksel dengan interpolasi bilinear dan noise Gaussian kecil diterapkan **hanya pada train**. Ini merupakan asumsi bahwa identitas angka masih dipertahankan pada rentang perturbasi tersebut. Pada aplikasi nyata, augmentasi harus diturunkan dari variasi proses akuisisi dan diuji apakah masih mempertahankan label.

**Larangan kebocoran:** jangan mengaugmentasi seluruh dataset sebelum pemisahan data. Citra hasil augmentasi dari sumber yang sama akan menyebabkan *near-duplicate leakage* bila tersebar ke train dan test.

In [ ]:
def train_augment(x):
    # translasi subpiksel acak, ±0,45 piksel pada tiap sumbu (NCHW)
    n=x.shape[0]
    theta=torch.zeros((n,2,3),dtype=x.dtype,device=x.device)
    theta[:,0,0]=1.0;theta[:,1,1]=1.0
    theta[:,0,2]=(torch.rand(n,device=x.device)-.5)*(.9/4.)
    theta[:,1,2]=(torch.rand(n,device=x.device)-.5)*(.9/4.)
    grid=F.affine_grid(theta,x.size(),align_corners=False)
    shifted=F.grid_sample(x,grid,mode='bilinear',padding_mode='zeros',align_corners=False)
    return torch.clamp(shifted+0.035*torch.randn_like(shifted),0,1)

original=torch.from_numpy(Xtr[:6]);aug=train_augment(original)
fig,axs=plt.subplots(2,6,figsize=(10,3.7))
for i in range(6):
    axs[0,i].imshow(original[i,0],cmap='gray',vmin=0,vmax=1);axs[0,i].axis('off')
    axs[1,i].imshow(aug[i,0],cmap='gray',vmin=0,vmax=1);axs[1,i].axis('off')
axs[0,0].set_ylabel('Asli');axs[1,0].set_ylabel('Train aug');
fig.suptitle('Augmentasi hanya pada pelatihan; bukan data baru yang independen');plt.tight_layout();plt.show()
assert not np.array_equal(original.numpy(),aug.numpy())

## 7. Protokol pelatihan dan *checkpoint* yang dapat diaudit

Model menerima tensor bentuk `(N,1,8,8)`, logit `(N,10)`. Optimizer AdamW, *cross-entropy*, seed terkontrol, batch tetap, dan early stopping dari validation loss. Setelah setiap epoch, semua sampel validation diprediksi **tanpa gradien**. Bobot terbaik disimpan sebagai salinan dalam memori. Test belum boleh dipakai dalam fungsi pelatihan.

Pelaporan minimal mencakup: fungsi objektif, optimizer, *learning rate*, *weight decay*, batas epoch, kriteria checkpoint, batch size, jumlah parameter, dan sistem komputasi. Perbandingan waktu CPU akan diukur namun tidak dapat digeneralisasikan ke GPU atau perangkat lain.

In [ ]:
def evaluate_net(model, x_np, y_np, batch=256):
    model.eval()
    x=torch.from_numpy(np.ascontiguousarray(x_np))
    parts=[]
    with torch.no_grad():
        for xx in x.split(batch): parts.append(model(xx.to(DEVICE)).cpu())
    logits=torch.cat(parts,dim=0)
    prob=F.softmax(logits,dim=1).numpy();pred=prob.argmax(axis=1)
    return {'pred':pred,'prob':prob,
            'f1':f1_score(y_np,pred,average='macro',zero_division=0),
            'acc':accuracy_score(y_np,pred),
            'nll':log_loss(y_np,prob,labels=list(range(10)))}

def train_one(kind='cnn', width=12, aug=False, seed=2026, epochs=18, patience=5):
    torch.manual_seed(seed);np.random.seed(seed)
    model=(TinyMLP() if kind=='mlp' else TinyCNN(width)).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=0.007,weight_decay=1e-3)
    xtr=torch.from_numpy(Xtr);yy=torch.from_numpy(ytr)
    ds=TensorDataset(xtr,yy)
    loader=DataLoader(ds,batch_size=96,shuffle=True,num_workers=0,
                      generator=torch.Generator().manual_seed(seed))
    best_val=float('inf');best_state=None;best_epoch=0;hist=[];wait=0
    t0=time.perf_counter()
    for epoch in range(1,epochs+1):
        model.train();loss_sum=0.0;n_seen=0
        for xx,labels in loader:
            xx,labels=xx.to(DEVICE),labels.to(DEVICE)
            if aug:xx=train_augment(xx)
            optimizer.zero_grad(set_to_none=True)
            loss=F.cross_entropy(model(xx),labels)
            loss.backward()
            optimizer.step()
            loss_sum+=float(loss.detach())*len(xx);n_seen+=len(xx)
        ev=evaluate_net(model,Xva,yva)
        hist.append({'epoch':epoch,'train_loss':loss_sum/n_seen,'val_loss':ev['nll'],'val_f1':ev['f1']})
        if ev['nll']<best_val-1e-4:
            best_val=ev['nll'];best_state=copy.deepcopy(model.state_dict())
            best_epoch=epoch;wait=0
        else:
            wait+=1
            if wait>=patience:break
    model.load_state_dict(best_state);model.eval()
    return {'model':model,'history':pd.DataFrame(hist),'best_epoch':best_epoch,
            'val':evaluate_net(model,Xva,yva),
            'train_sec':time.perf_counter()-t0, 'n_params':nparams(model),
            'kind':kind,'width':width,'aug':aug,'seed':seed}

print('Rancangan fungsi pelatihan siap. Test belum dipakai.')

## 8. Eksperimen utama: baseline, MLP, CNN kecil, CNN, dan CNN + augmentasi

Pelatihan dilakukan dengan seed sama pada satu split untuk mengurangi variasi yang tidak perlu. Setiap model memilih *checkpoint* dari validation loss, dan performa test **belum dihitung pada tahap ini**.

Hati-hati mengartikan *ablation*: perbedaan arsitektur mengubah kapasitas parameter dan *inductive bias* sekaligus; perbandingan tersebut adalah deskriptif. Perbandingan augmentasi dengan lebar model tetap lebih terkendali, tetapi akan lebih kuat jika diulang pada banyak seed/split atau data lapangan independen.

In [ ]:
experiments={}
settings=[('MLP',{'kind':'mlp','aug':False}),
          ('CNN kecil',{'kind':'cnn','width':6,'aug':False}),
          ('CNN',{'kind':'cnn','width':12,'aug':False}),
          ('CNN + augmentasi',{'kind':'cnn','width':12,'aug':True})]
for name,kwargs in settings:
    experiments[name]=train_one(**kwargs,seed=SEED)
    r=experiments[name]
    print(f'{name:18s} | epoch={r["best_epoch"]:2d} | val macro-F1={r["val"]["f1"]:.3f} '
          f'| parameter={r["n_params"]:5d} | CPU train={r["train_sec"]:.2f}s')

### 8.1 Kurva belajar: *underfitting*, *overfitting*, dan fluktuasi

Kurva train dan validation membantu mengidentifikasi proses optimisasi. Perhatikan bahwa nilai training dihitung pada data yang telah dilihat dalam epoch tersebut, sedangkan validation dievaluasi setelah pembaruan parameter. Selisih kurva tidak selalu dapat dibaca sebagai estimator bias model karena augmentasi dan perbedaan prosedur pengukuran.

Tidak selalu benar bahwa model dengan training loss paling rendah mempunyai ketahanan terbaik. Kita memerlukan penilaian terhadap data di luar pelatihan, terutama ketika citra berasal dari proses akuisisi berbeda.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,3.7))
for name,r in experiments.items():
    h=r['history'];axs[0].plot(h.epoch,h.val_loss,marker='.',label=name)
    axs[1].plot(h.epoch,h.val_f1,marker='.',label=name)
axs[0].set(xlabel='Epoch',ylabel='Validation NLL',title='Loss validation')
axs[1].set(xlabel='Epoch',ylabel='Macro-F1 validation',title='Performa validation')
axs[0].legend(fontsize=8);plt.tight_layout();plt.show()

## 9. Memilih konfigurasi berdasarkan validasi, bukan *test*

Untuk menghindari pemenang yang dipilih setelah melihat test, gunakan aturan yang ditetapkan sejak awal: **macro-F1 validation terbesar**, kemudian validation NLL paling kecil jika ada seri. Kita mempertahankan SVM sebagai baseline untuk perbandingan akhir; ia juga memakai validation set untuk memilih $C$.

Karena semua model bersaing pada validation set yang sama, skor validation model terbaik memiliki *winner's curse*. Hasil test hanya menjadi estimasi satu kali pada data tertahan, bukan pembuktian keunggulan universal.

In [ ]:
validation_table=pd.DataFrame([{'model':name,'val_f1':r['val']['f1'],'val_nll':r['val']['nll'],
                                 'parameter':r['n_params'],'waktu_train_s':r['train_sec']}
                                for name,r in experiments.items()])
validation_table.loc[len(validation_table)]={'model':'SVM piksel','val_f1':best_svm_val,
    'val_nll':log_loss(yva,best_svm.predict_proba(Xva.reshape(len(Xva),-1)),labels=list(range(10))),
    'parameter':np.nan,'waktu_train_s':svm_time}
validation_table=validation_table.sort_values(['val_f1','val_nll'],ascending=[False,True]).reset_index(drop=True)
selected_name=str(validation_table.iloc[0]['model'])
print(validation_table.round(4).to_string(index=False))
print('\nModel terpilih berdasarkan validation:',selected_name)
assert len(Xte)==len(yte)==360

## 10. Evaluasi akhir pada *held-out test*: metrik dan kesalahan kelas

Setelah konfigurasi tetap, lakukan **satu tahap evaluasi final** pada test. Metrik yang relevan meliputi macro-F1 (memperlakukan kelas setara), akurasi total, negatif log-likelihood, confusion matrix, dan pemeriksaan kelas yang sering tertukar. Karena test berukuran terbatas, perbedaan beberapa sampel dapat menggeser macro-F1.

Untuk keterbacaan, ditampilkan semua model yang telah dipilih berdasarkan prosedur identik, tetapi tabel test **tidak boleh digunakan untuk melakukan penyetelan ulang** lalu dilaporkan seolah-olah data itu masih tertahan.

In [ ]:
test_results={}
for name,r in experiments.items():test_results[name]=evaluate_net(r['model'],Xte,yte)
svm_probs=best_svm.predict_proba(Xte.reshape(len(Xte),-1))
svm_preds=best_svm.predict(Xte.reshape(len(Xte),-1))
test_results['SVM piksel']={'prob':svm_probs,'pred':svm_preds,
    'f1':f1_score(yte,svm_preds,average='macro'), 'acc':accuracy_score(yte,svm_preds),
    'nll':log_loss(yte,svm_probs,labels=list(range(10)))}
test_table=pd.DataFrame([{'model':k,'test_macro_f1':v['f1'],'test_accuracy':v['acc'],'test_nll':v['nll']}
                         for k,v in test_results.items()]).sort_values('test_macro_f1',ascending=False)
print(test_table.round(4).to_string(index=False))

In [ ]:
chosen=test_results[selected_name]
cm=confusion_matrix(yte,chosen['pred'],labels=list(range(10)))
fig,axs=plt.subplots(1,2,figsize=(10,4))
axs[0].imshow(cm,cmap='Blues');axs[0].set(xlabel='Prediksi',ylabel='Label',title='Confusion matrix model terpilih')
for i in range(10):
 for j in range(10):
    if cm[i,j]>0:axs[0].text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=7)
axs[1].bar(test_table['model'],test_table['test_macro_f1']);axs[1].set_ylim(0,1)
axs[1].set(title='Test macro-F1',ylabel='Macro-F1');axs[1].tick_params(axis='x',rotation=45)
plt.tight_layout();plt.show()
print('Laporan model terpilih:\n',classification_report(yte,chosen['pred'],zero_division=0,digits=3))

## 11. Interpretabilitas: aktivasi, saliency, dan ablation piksel

Interpretabilitas memerlukan kehati-hatian epistemik. Aktivasi kanal dapat menunjukkan respons terhadap pola lokal, tetapi **tidak membuktikan** sebab keputusan model. *Input gradient saliency* mengukur sensitivitas lokal keluaran terhadap masukan; ia bisa berisik, berubah karena skala input, dan bukan bukti sebab-akibat.

Sebagai triangulasi sederhana, kita bandingkan citra asli, peta aktivasi, gradien terhadap input, dan **occlusion map** (perubahan logit ketika satu piksel dihilangkan). Occlusion memodifikasi distribusi input dan bukan intervensi fisik yang otomatis sah. Karena citra Digits hanya 8×8, seluruh peta beresolusi rendah.

In [ ]:
# Interpretabilitas khusus CNN; ambil CNN yang dilatih, bukan mengganti model terpilih tanpa catatan.
cnn_interpret=experiments['CNN']['model'];cnn_interpret.eval()
example_id=int(np.flatnonzero(yte==8)[0]);x_example=torch.from_numpy(Xte[example_id:example_id+1].copy()).requires_grad_(True)
activation={}
def hook_fn(module,inp,out):activation['conv1']=out.detach().cpu().numpy()
handle=cnn_interpret.features[0].register_forward_hook(hook_fn)
logits=cnn_interpret(x_example);label=int(logits.argmax(dim=1)[0]);score=logits[0,label]
cnn_interpret.zero_grad(set_to_none=True);score.backward();saliency=np.abs(x_example.grad.detach().numpy()[0,0]);handle.remove()
conv_maps=activation['conv1'][0]
fig,axs=plt.subplots(2,4,figsize=(11,5))
axs[0,0].imshow(Xte[example_id,0],cmap='gray');axs[0,0].set_title('Input kelas '+str(yte[example_id]))
axs[0,1].imshow(saliency,cmap='magma');axs[0,1].set_title('Saliency |gradien|')
for j in range(6):axs.flat[j+2].imshow(conv_maps[j],cmap='viridis');axs.flat[j+2].set_title(f'Conv1 map {j}')
for ax in axs.flat:ax.axis('off')
plt.tight_layout();plt.show()

In [ ]:
# Occlusion satu piksel: perubahan logit kelas prediksi; sensitivitas, bukan interpretasi kausal mutlak.
x0=torch.from_numpy(Xte[example_id:example_id+1].copy())
with torch.no_grad():base=float(cnn_interpret(x0)[0,label])
occlusion=np.zeros((8,8))
with torch.no_grad():
 for i in range(8):
  for j in range(8):
   xx=x0.clone();xx[0,0,i,j]=0
   occlusion[i,j]=base-float(cnn_interpret(xx)[0,label])
fig,ax=plt.subplots(1,2,figsize=(7,3))
ax[0].imshow(x0[0,0],cmap='gray');ax[0].set_title('Citra referensi')
im=ax[1].imshow(occlusion,cmap='coolwarm');ax[1].set_title('Δ logit akibat oklusi')
for a in ax:a.axis('off')
fig.colorbar(im,ax=ax[1],fraction=.047);plt.tight_layout();plt.show()

## 12. Robustness terhadap perubahan proses pencitraan

Tiga gangguan buatan diterapkan **pada seluruh citra test yang sama**, bukan pada train: (i) noise Gaussian pada intensitas, (ii) blur spasial, (iii) translasi kecil, dan (iv) kombinasi ketiganya. Parameter ditetapkan sebelum menghitung metrik; tidak disesuaikan sesudah melihat hasil stress.

Perbandingan ini adalah pengujian sensitivitas terbatas. *Domain shift* nyata bisa meliputi karakteristik spektral sensor, demosaicing, ukuran objek, kompresi, protokol anotasi, dan perubahan prevalensi kelas. Tidak boleh menganggap manipulasi sintetis sebagai bukti robust terhadap semua perubahan tersebut.

In [ ]:
def stress_images(x,kind,seed=991):
    rng=np.random.default_rng(seed)
    out=np.array(x,dtype=np.float32,copy=True)
    if kind in ('blur','gabungan'):
        out=gaussian_filter(out,sigma=(0,0,0.6,0.6)).astype(np.float32)
    if kind in ('shift','gabungan'):
        # translasi yang sama untuk semua sampel, tanpa wrap-around
        out=scipy_shift(out,shift=(0,0,0.65,-0.60),order=1,mode='constant',cval=0).astype(np.float32)
    if kind in ('noise','gabungan'):
        out+=rng.normal(0,0.13 if kind=='noise' else 0.08,size=out.shape).astype(np.float32)
    return np.clip(out,0,1)

stress_sets={'normal':Xte,**{k:stress_images(Xte,k) for k in ['noise','blur','shift','gabungan']}}
fig,axs=plt.subplots(1,5,figsize=(11,2.8))
for ax,(name,xv) in zip(axs,stress_sets.items()):
    ax.imshow(xv[2,0],cmap='gray',vmin=0,vmax=1);ax.set_title(name);ax.axis('off')
fig.suptitle('Gangguan buatan pada citra test yang sama');plt.tight_layout();plt.show()

In [ ]:
robust_rows=[]
for name in list(experiments)+['SVM piksel']:
    for condition,xx in stress_sets.items():
        if name=='SVM piksel':
            p=best_svm.predict(xx.reshape(len(xx),-1))
        else:p=evaluate_net(experiments[name]['model'],xx,yte)['pred']
        robust_rows.append({'model':name,'kondisi':condition,'macro_f1':f1_score(yte,p,average='macro',zero_division=0)})
robust_table=pd.DataFrame(robust_rows)
print(robust_table.pivot(index='model',columns='kondisi',values='macro_f1').round(3).to_string())
fig,ax=plt.subplots(figsize=(10,4))
for model,g in robust_table.groupby('model',sort=False):
    ax.plot(g.kondisi,g.macro_f1,marker='o',label=model)
ax.set(xlabel='Kondisi',ylabel='Macro-F1 test yang sama',title='Sensitivitas model terhadap perturbasi citra');ax.set_ylim(0,1)
ax.legend(fontsize=8);plt.tight_layout();plt.show()

### 12.1 *Gradient-based stress* bukan metrik kemampuan umum

Sebuah perturbasi FGSM (Fast Gradient Sign Method) mengubah masukan berdasarkan tanda gradien loss: $x'=\operatorname{clip}(x+\varepsilon\,\mathrm{sign}(\nabla_x\mathcal{L}),0,1)$. Ini menghasilkan gangguan **bergantung pada model**. Eksperimen hanya menunjukkan satu tipe sensitivitas dengan batas $\varepsilon$ tertentu; bukan pengganti pengujian perangkat nyata ataupun bukti sistem aman/tidak aman pada semua situasi.

In [ ]:
# Demonstrasi FGSM pada 40 sampel, tidak digunakan untuk seleksi model.
model_fgsm=experiments['CNN']['model'];model_fgsm.eval()
x_adv=torch.from_numpy(Xte[:40].copy()).requires_grad_(True)
y_adv=torch.from_numpy(yte[:40])
model_fgsm.zero_grad(set_to_none=True)
F.cross_entropy(model_fgsm(x_adv),y_adv).backward()
with torch.no_grad():
    x_attack=(x_adv+0.08*x_adv.grad.sign()).clamp(0,1)
    p_clean=model_fgsm(torch.from_numpy(Xte[:40])).argmax(1).numpy()
    p_attack=model_fgsm(x_attack).argmax(1).numpy()
print('Akurasi pada 40 citra bersih:',round(float(np.mean(p_clean==yte[:40])),3))
print('Akurasi pada 40 citra FGSM:',round(float(np.mean(p_attack==yte[:40])),3))
fig,ax=plt.subplots(1,3,figsize=(8,2.6))
for a,xx,t in zip(ax,[Xte[0,0],x_attack.detach().numpy()[0,0],x_attack.detach().numpy()[0,0]-Xte[0,0]],
                  ['Bersih','FGSM ε=0,08','Δ input']):
    a.imshow(xx,cmap='gray' if t!='Δ input' else 'coolwarm');a.axis('off');a.set_title(t)
plt.tight_layout();plt.show()

## 13. Kalibrasi probabilitas: confidence tidak sama dengan kebenaran

Prediksi kelas adalah $\hat y=\arg\max_k p_k$, tetapi *confidence* $c=\max_k p_k$ memerlukan audit. Model dapat mengeluarkan confidence tinggi ketika salah, terutama pada shift domain. **Expected Calibration Error (ECE)** tergantung jumlah/posisi bin dan bersifat ringkasan; ia tidak menguji semua aspek distribusi prediktif. *Negative log-likelihood* dan *Brier score* bersifat *proper scoring rules* untuk evaluasi probabilitas.

Untuk kelas banyak, Brier multikelas adalah

$$
B=\frac1N\sum_{i=1}^N\sum_{k=1}^K(p_{ik}-\mathbb{1}[y_i=k])^2.
$$

Kalibrasi temperatur harus disetel pada validasi jika diterapkan; karena fokus di sini adalah diagnosis, tidak dilakukan fitting kalibrator pada test.

In [ ]:
def calibration_stats(y_true, probs, n_bins=10):
    conf=probs.max(axis=1);pred=probs.argmax(axis=1);ok=(pred==y_true).astype(float)
    edges=np.linspace(0,1,n_bins+1);ece=0.;rows=[]
    ids=np.minimum(np.digitize(conf,edges[1:-1],right=False),n_bins-1)
    for b in range(n_bins):
        mask=ids==b
        if mask.any():
            a=float(ok[mask].mean()); c=float(conf[mask].mean())
            ece+=mask.mean()*abs(a-c);rows.append((b,int(mask.sum()),c,a))
    hot=np.eye(probs.shape[1])[y_true]
    return float(ece),float(np.mean(np.sum((probs-hot)**2,axis=1))),pd.DataFrame(rows,columns=['bin','n','confidence','accuracy'])

cal_rows=[]
for name,r in test_results.items():
    ece,brier,_=calibration_stats(yte,r['prob'])
    cal_rows.append({'model':name,'ECE_10bin':ece,'Brier_multi':brier,'NLL':r['nll']})
print(pd.DataFrame(cal_rows).round(4).to_string(index=False))
_,_,reliability=calibration_stats(yte,chosen['prob'])
fig,ax=plt.subplots(figsize=(4.5,4))
ax.plot([0,1],[0,1],'k--',label='Kalibrasi ideal')
ax.plot(reliability.confidence,reliability.accuracy,'o-',label=selected_name)
ax.set(xlim=(0,1),ylim=(0,1),xlabel='Confidence rata-rata per bin',ylabel='Akurasi per bin',title='Reliability (bin tidak sama banyak)')
ax.legend();plt.tight_layout();plt.show()

## 14. Efisiensi komputasi dan kesetaraan pembandingan

Jumlah parameter **bukan** FLOPs, RAM, latensi, ataupun konsumsi energi. Pengukuran waktu sangat dipengaruhi prosesor, thread, keadaan cache, *warm-up*, dan batch size. Catatan waktu dari lingkungan ini berlaku **hanya** pada sistem eksekusi yang dilaporkan. Untuk studi engineering, latency dan throughput harus diukur pada perangkat target dengan pengulangan dan interval kepercayaan.

Sebagai demonstrasi, gunakan 5 pengulangan latensi inferensi seluruh test pada CPU, setelah satu *warm-up*. Catatan ini **tidak** membandingkan efisiensi energi atau emisi.

In [ ]:
latency=[]
for name,r in experiments.items():
    model=r['model'];model.eval();xx=torch.from_numpy(Xte)
    with torch.no_grad():_ = model(xx[:32])
    seconds=[]
    for _ in range(5):
        t0=time.perf_counter()
        with torch.no_grad():_ = model(xx)
        seconds.append(1000*(time.perf_counter()-t0)/len(xx))
    latency.append({'model':name,'parameter':r['n_params'],'train_s':r['train_sec'],
                    'ms_per_sample_batch360_median':float(np.median(seconds))})
print(pd.DataFrame(latency).round(4).to_string(index=False))

## 15. Ketidakpastian inferensi: *paired bootstrap* dan pengulangan seed

Karena setiap model diuji pada **citra yang sama**, perbedaan macro-F1 harus dievaluasi dengan resampling **berpasangan** pada indeks gambar. Kita membandingkan CNN dan SVM yang telah ditentukan dalam rancangan; interval persentil dihitung dari 500 sampel bootstrap. Interval ini mendeskripsikan variasi akibat resampling citra **dengan prediksi model tetap**. Ia tidak memuat variasi penulis, perangkat, seed pelatihan, atau protokol akuisisi.

Evaluasi seharusnya diulang pada split atau domain independen untuk mendukung generalisasi yang lebih luas. Interval yang memuat nol tidak otomatis membuktikan kedua model setara.

In [ ]:
bootstrap_rng=np.random.default_rng(137)
A=test_results['CNN']['pred'];B=test_results['SVM piksel']['pred']
N=len(yte);diffs=[]
for _ in range(500):
    ii=bootstrap_rng.integers(0,N,N)
    da=f1_score(yte[ii],A[ii],average='macro',zero_division=0)
    db=f1_score(yte[ii],B[ii],average='macro',zero_division=0)
    diffs.append(da-db)
lo,hi=np.percentile(diffs,[2.5,97.5])
print(f'Delta macro-F1 CNN − SVM = {f1_score(yte,A,average="macro")-f1_score(yte,B,average="macro"):+.4f}')
print(f'Paired bootstrap 95% percentile interval (bersyarat pada split/model): [{lo:+.4f}, {hi:+.4f}]')
fig,ax=plt.subplots(figsize=(6.5,3))
ax.hist(diffs,bins=25,color='slateblue',alpha=.8);ax.axvline(0,color='black',ls='--')
ax.axvline(lo,color='crimson',ls=':');ax.axvline(hi,color='crimson',ls=':')
ax.set(xlabel='Perbedaan macro-F1',ylabel='Frekuensi bootstrap',title='Resampling berpasangan pada test');plt.tight_layout();plt.show()

### 15.1 Sensitivitas seed pada pelatihan CNN

Dua pelatihan tambahan dengan **seed berbeda** dilakukan pada split tetap. Tujuannya mengilustrasikan keragaman optimisasi, bukan menghasilkan estimasi ketidakpastian antarpopulasi. Test tidak dipakai dalam pemilihan checkpoint tiap seed. Untuk laporan riset yang kuat, jumlah pengulangan dan pembagian data sebaiknya ditentukan sebelum eksperimen dan diperluas sesuai sumber daya.

In [ ]:
seed_runs=[]
for sd in [2027,2028]:
    run=train_one('cnn',width=12,aug=False,seed=sd,epochs=18,patience=5)
    seed_runs.append({'seed':sd,'epoch_terpilih':run['best_epoch'],
                      'validation_f1':run['val']['f1'],
                      'test_f1_evaluasi_akhir':evaluate_net(run['model'],Xte,yte)['f1']})
seed_runs.insert(0,{'seed':SEED,'epoch_terpilih':experiments['CNN']['best_epoch'],
    'validation_f1':experiments['CNN']['val']['f1'],'test_f1_evaluasi_akhir':test_results['CNN']['f1']})
print(pd.DataFrame(seed_runs).round(4).to_string(index=False))

## 16. Audit implementasi dan hasil

Kepercayaan ilmiah pada hasil membutuhkan pemeriksaan numerik **dan** alur data. Uji berikut memeriksa: isolasi split, penjumlahan probabilitas, jumlah parameter, turunan cross-entropy, kesetaraan hasil korelasi manual, rentang citra, invariansi jumlah label, dan bahwa stress berasal dari test yang sama. Uji ini bukan bukti verifikasi lengkap pada semua perangkat atau versi pustaka.

**Perhatikan:** Anda dapat membandingkan hasil dengan notebook 09, tetapi hanya jika split, pemrosesan, skema tuning, dan metrik benar-benar sama. Jika protokol berbeda, angka F1 tidak dapat dibandingkan langsung sebagai *fair benchmark*.

In [ ]:
checks={
 'train-val-test disjoint': not(set(idx_train)&set(idx_val) or set(idx_train)&set(idx_test) or set(idx_val)&set(idx_test)),
 'NCHW shape': Xtr.shape[1:]==(1,8,8),
 'nilai input 0–1': bool(np.all((Xtr>=0)&(Xtr<=1))),
 'logit output 10 kelas': experiments['CNN']['model'](torch.zeros(2,1,8,8)).shape==(2,10),
 'softmax jumlah 1': bool(np.allclose(chosen['prob'].sum(axis=1),1,atol=1e-5)),
 'test tidak masuk training': len(set(idx_train)&set(idx_test))==0,
 'tabel stress berpasangan': all(len(v)==len(Xte) for v in stress_sets.values()),
 'identitas sampel test': len(yte)==len(chosen['pred'])==360,
 'kernel manual setara': bool(np.isclose(float(hasil[0,0,0,0]),manual)),
 'gradien CE setara':bool(np.allclose(logit.grad.numpy().ravel(),analitik,atol=1e-10)),
}
print(pd.Series({k:'LULUS' if v else 'GAGAL' for k,v in checks.items()}).to_string())
assert all(checks.values())

## 17. Studi kasus integratif: sistem pembacaan angka pada kondisi akuisisi berubah

**Pertanyaan ilmiah:** kapan CNN layak digunakan dibanding metode klasik jika perangkat harus membaca angka kecil dengan noise dan blur? Rancang keputusan berdasarkan tujuan sistem, bukan hanya *leaderboard*.

**Desain penelitian yang telah dijalankan:**

1. Definisikan masalah dan asumsi digit 8×8.
2. Pisahkan train/validation/test dengan seed dan stratifikasi terdokumentasi.
3. Gunakan SVM sebagai baseline, MLP sebagai kontrol tanpa konvolusi, dan dua kapasitas CNN.
4. Ubah hanya augmentasi dalam ablation CNN utama; pilih checkpoint berdasarkan validation loss.
5. Pilih satu konfigurasi berdasarkan validation F1, lalu hitung test.
6. Audit confusion matrix, confidence, perturbasi, latensi, dan interval bootstrap.
7. Laporkan unit eksperimen, keterbatasan data, dan batas klaim.

**Keputusan engineering:** Jika kesalahan salah-baca sangat mahal, pemilihan model memerlukan matriks biaya kesalahan, threshold penolakan prediksi, uji kamera target, dan protokol kalibrasi. Nilai probabilitas softmax tidak boleh langsung digunakan untuk klaim risiko tanpa kalibrasi dan verifikasi eksternal.

In [ ]:
# Ringkasan yang dapat disalin ke mini-laporan, seluruh nilai berasal dari eksekusi notebook.
summary={
 'dataset':'scikit-learn Digits (8x8, 10 kelas)',
 'split':{'train':len(idx_train),'validation':len(idx_val),'test':len(idx_test)},
 'model_validasi_terpilih':selected_name,
 'hasil_test_macro_f1':round(float(chosen['f1']),5),
 'hasil_stress_gabungan_macro_f1':round(float(robust_table.query('model==@selected_name and kondisi=="gabungan"').iloc[0]['macro_f1']),5),
 'bootstrap_CNN_minus_SVM_CI95':[round(float(lo),5),round(float(hi),5)],
 'torch':torch.__version__, 'sklearn':sklearn.__version__, 'seed':SEED,
 'catatan':'Hasil satu lingkungan, bukan pengujian sensor/industri; bootstrap mengondisikan model dan split.'
}
print(json.dumps(summary,ensure_ascii=False,indent=2))

## 18. Latihan konseptual, komputasional, dan HOTS

**A. Konseptual dan derivasi**

1. **Konvolusi dan korelasi.** Turunkan keluaran kernel 3×3 pada patch 5×5 dan jelaskan kapan pembalikan kernel mengubah hasil. Jelaskan peran *padding*.
2. **Dimensi dan biaya.** Hitung bentuk tensor, jumlah parameter, dan *receptive field* untuk dua blok konvolusi dengan stride berbeda. Bedakan parameter dan FLOPs.
3. **Gradien.** Derivasikan $\partial\mathcal L/\partial z_k=p_k-y_k$ dan hubungkan dengan gradien bobot kernel melalui aturan rantai.
4. **Pooling.** Berikan contoh citra dua kelas yang berbeda hanya pada koordinat fitur; tunjukkan kapan pooling membuang informasi kelas.

**B. Eksperimen dengan hipotesis**

5. **Ablasi kanal.** Ubah lebar CNN menjadi 4, 8, dan 16. Periksa *training time*, jumlah parameter, validation F1, dan stress F1; gunakan seed dan split yang sama.
6. **Ablasi augmentasi.** Pisahkan dampak translasi saja, noise saja, dan kombinasi. Terapkan augmentasi train saja. Jangan menyeleksi augmentasi berdasarkan test.
7. **Variasi ukuran batch.** Bandingkan batch 32, 96, 256 dengan laju pembelajaran tetap dan terkontrol. Laporkan perubahan jumlah update dan konsekuensi fairness.
8. **Kalibrasi.** Terapkan temperature scaling yang dipilih hanya dari validation. Nilai perubahan NLL, Brier, dan ECE test; jelaskan batas ECE.

**C. Penalaran tingkat lanjut**

9. **Analisis lemah pada Digits.** Identifikasi tiga sebab skor Digits tidak boleh digeneralisasikan ke kamera smartphone dengan teks latar kompleks. Rancang protokol data baru.
10. **Interpretabilitas kritis.** Bandingkan gradien, occlusion, dan aktivasi. Buat contoh alasan mengapa peta panas tidak membuktikan mekanisme kausal.
11. **Statistik pembandingan.** Rancang studi multisplit/multiseed, tentukan unit independen, dan jelaskan interval bootstrap yang sesuai jika ada banyak gambar dari satu penulis.
12. **Keputusan metrologi.** Pertimbangkan kasus di mana jaringan memprediksi kelas benar tetapi kehilangan informasi ukuran, bentuk, atau posisi tepi. Apakah model itu baik untuk inspeksi dimensi? Rancang metrik pengukuran fisik.

**Ketentuan laporan:** tujuan dan asumsi → prosedur dan *split* → tabel metrik → plot → analisis kesalahan → keterbatasan → kesimpulan yang sesuai bukti. Setiap klaim tentang perangkat nyata harus didukung pengujian perangkat nyata.

## 19. Rangkuman sintesis: apa yang benar-benar dipelajari?

- CNN menerapkan filter berbagi bobot dan transformasi nonlinear, tetapi keunggulannya dipengaruhi data, arsitektur, dan tujuan evaluasi.
- *Validation-based selection* penting agar test tidak menjadi alat tuning terselubung.
- Augmentasi harus memperkirakan variasi fisik yang labelnya tetap; augmentasi sembarang berpotensi merusak target tugas.
- *Saliency* dan aktivasi adalah diagnosis, bukan sertifikat penjelasan model.
- Metrik probabilitas, *domain shift*, ketidakpastian sampling, dan biaya komputasi memperluas evaluasi di luar akurasi.
- Hasil komputasi pada Digits tidak boleh disamakan dengan kemampuan membaca citra kamera nyata, apalagi sistem kritis keselamatan.

**Koneksi berikutnya:** pembentukan citra dan *inverse problems* mengaitkan model sensor, PSF, noise, dan dekonvolusi; optimisasi memperdalam fungsi objektif serta regularisasi; Bayesian imaging menilai posterior dan ketidakpastian; *domain shift* dan desain eksperimen menguji apakah hasil dapat dipercaya di kondisi akuisisi berbeda. Semua bab tersebut memperluas analisis CNN menjadi penelitian pencitraan yang dapat dipertanggungjawabkan.

## 20. Referensi dan pedoman membaca

1. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. Bab 6–9. https://www.deeplearningbook.org/
2. LeCun, Y., Bottou, L., Bengio, Y., & Haffner, P. (1998). Gradient-based learning applied to document recognition. *Proceedings of the IEEE*, 86(11), 2278–2324. DOI: 10.1109/5.726791.
3. Srivastava, N., et al. (2014). Dropout: A Simple Way to Prevent Neural Networks from Overfitting. *JMLR*, 15, 1929–1958.
4. Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. (2017). On Calibration of Modern Neural Networks. *ICML*. https://proceedings.mlr.press/v70/guo17a.html
5. PyTorch. `torch.nn.Conv2d`, `CrossEntropyLoss`, dan `AdamW` (dokumentasi API): https://pytorch.org/docs/stable/nn.html
6. scikit-learn. `load_digits` dan evaluasi klasifikasi: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html
7. Koh, P. W., et al. (2021). WILDS: A Benchmark of in-the-Wild Distribution Shifts. *ICML*. https://proceedings.mlr.press/v139/koh21a.html
8. Adebayo, J., et al. (2018). Sanity Checks for Saliency Maps. *NeurIPS*. https://proceedings.neurips.cc/paper/2018/hash/294a8ed24b1ad22ec2e7efea049b8737-Abstract.html

Bacaan di atas merupakan referensi konsep dan metodologi; angka eksperimen notebook diperoleh dari eksekusi dan tidak dikutip sebagai hasil dari literatur tersebut.